In [ ]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Selecting batch size

In [2]:
batch_size = 50


Zero-shot prompt tempalte for 2 label task: Hate + Neutral

In [4]:
prompt_template = '''You are tasked with annotating speech. Your response must be a single valid number:
    1 for Hate Speech.
    2 for Offensive Speech.
    3 for Neutral Speech.

    Think step by step:
    - Identify whether the comment contains explicit hate towards a group or individual based on identity characteristics (Hate Speech).
    - If the comment is aggressive, rude, or contains insults but lacks clear hateful intent, classify it as Offensive Speech.
    - If the comment is neutral or non-offensive, classify it as Neutral Speech.

    Examples:
    Comment: "People of [group] are a disease and should be removed."
    Your Answer: 1

    Comment: "You're so stupid, just shut up!"
    Your Answer: 2

    Comment: "I hope you have a great day!"
    Your Answer: 3

    Provide only the number corresponding to the category. Do not include any explanation or additional text or your thinking process.
    Do you think the following comment is hate speech or offensive speech?
    \n"{comment}"\n
    Your Answer:
    '''



List of finetuned models available in Hugging Face. However, for anonymity, only the base versions—using the Unsloth models—are show here.

In [ ]:
models_list_HF = {
                    "Base": "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
                    "Human": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Human.3_label",
                    "Lgb": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Lgb.3_label",
                    "Mean": "anonymousOWSHateLLM/Hate-Qwen2.5-14B.Mean.3_label",
                    }


Evaluation set: Two group 1 and group 2

In [26]:
df_eval_set = pd.read_csv("df_eval_set.csv")
df_eval_set.loc[0]

Unnamed: 0                                                             0
text                   Notwithstanding Marriyum Aurangzeb sahiba's po...
ds                                                              berkeley
label_id                                                               3
llama_70B                                                              3
prompt                 <|im_start|>system\nYou are Qwen, created by A...
y_ture                                                                 3
y_pred                                                                 3
Base_probs_label_1                                                   0.0
Base_probs_label_2                                                0.0015
Base_probs_label_3                                                   0.0
Human_probs_label_1                                              0.00433
Human_probs_label_2                                              0.05981
Human_probs_label_3                                

In [ ]:

dataset = load_dataset("")

In [5]:
dataset

DatasetDict({
    val_3_label: Dataset({
        features: ['Unnamed: 0', 'text', 'ds', 'label_id', 'llama_70B'],
        num_rows: 5241
    })
})

In [10]:
df_eval_set = pd.DataFrame(dataset["val_3_label"])
print("Total Samples of set 1: ",df_eval_set.shape[0])
print(df_eval_set.groupby(['ds', 'label_id']).size().unstack(fill_value=0))

Total Samples of set 1:  5241
label_id       1    2    3
ds                        
HateOff       62  990  193
HateSpeechX  334  265  397
SetFit       500  500  500
berkeley     500  500  500


Selecting Group model Llama3.2-1B or Qwen2.5-14B 
Test set: 1 or 2

In [7]:
def process_task(texts, model, tokenizer, stop_token_id):
    encoding = tokenizer(texts, padding=True, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model(**encoding)
        logits = outputs.logits  
    last_token_logits = logits[:, -1, :] 
    probabilities = torch.softmax(last_token_logits, dim=-1)
    indices = torch.tensor(stop_token_id)
    probs = []
    for i in indices:
        probs.append( probabilities[:, i].float().cpu().numpy())
    return probs


In [8]:
def preprocess(text, model_id, tokenizer):
    user_message_content = prompt_template.format(comment=text)
    user_message = {
        "role": "user",
        "content": user_message_content
    }

    if "Qwen" in model_id:
        system_message =  {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant"}
    else:
        system_message =  {"role": "system", "content": "You are a helpful assistant"}
    messages = [system_message, user_message]
    messages = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    messages = messages


    return messages


In [11]:
df_eval_set.columns

Index(['Unnamed: 0', 'text', 'ds', 'label_id', 'llama_70B'], dtype='object')

In [12]:
def run_eval():
    model_list = models_list_HF
    df_eval = df_eval_set
    model_probs_dict = {}
    model_probs_dict['Llama3.1-70B'] = {}
    model_probs_dict['Llama3.1-70B']['probs'] = df_eval['llama_70B']
    for key, model_id in model_list.items():

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_id,
            max_seq_length=500,
            dtype=getattr(torch, "bfloat16"),
        )
        FastLanguageModel.for_inference(model)
        tokenizer.padding_side = "left"


        stop_token_id = tokenizer(["123"])['input_ids'][0]


        df_eval["prompt"] = df_eval["text"].apply(lambda text: preprocess(text, model_id, tokenizer))


        texts = []
        probs = []
        for i in range(len(stop_token_id)):
            probs.append([])

        prompts = df_eval['prompt'].tolist()

        for i in tqdm(range(0, len(prompts), batch_size)):
            batch = prompts[i:i+batch_size]
            prob_return = process_task(batch, model, tokenizer, stop_token_id)
            for i2, p in enumerate(probs):
                probs[i2] += prob_return[i2].tolist()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()


        model_probs_dict[key] = {
            "probs": probs,
        }

    report = {}
    for ds in df_eval['ds'].unique():
        report[ds] = {}
    report["Mean"] = {}

    y_true = df_eval['label_id']
    df_eval['y_ture'] = y_true

    for model, probs in model_probs_dict.items():
        if model == 'Llama3.1-70B':
            df_eval['y_pred'] = probs["probs"]
        else:
            probs = probs["probs"]


            probabilities = np.array([probs[0], probs[1], probs[2]]).T  

            df_eval['y_pred'] = np.argmax(probabilities, axis=1) + 1

        report["Mean"][model] = {
        "acc": round(accuracy_score(y_true, df_eval['y_pred'])* 100, 1), 
        "f1": round(f1_score(y_true,df_eval['y_pred'], average="macro")* 100, 1)
            }
        
        for ds in df_eval['ds'].unique():
            tmp_df = df_eval.loc[df_eval['ds'] == ds]

            
            y_pred = tmp_df['y_pred']
            acc = round(accuracy_score(tmp_df['y_ture'],y_pred)* 100, 1) 
            f1 = round(f1_score(tmp_df['y_ture'], y_pred, average= "macro")* 100, 1) 
            report[ds][model] = {
                            "acc": acc, 
                            "f1": f1
                            }

    report_df = pd.DataFrame(report).T
    return model_probs_dict, report_df

In [13]:
probs, report = run_eval()

report


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████| 105/105 [04:28<00:00,  2.56s/it]


==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

...

==((====))==  Unsloth 2025.2.5: Fast Qwen2 patching. Transformers: 4.48.3.
   \\   /|    GPU: NVIDIA A100 80GB PCIe. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29. FA2 = True]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/138M [00:00<?, ?B/s]

100%|█████████████████████████████████████████████████████████████| 105/105 [05:09<00:00,  2.95s/it]


,Llama3.1-70B,Base,Human,Lgb,Mean
berkeley,"{'acc': 74.6, 'f1': 74.3}","{'acc': 75.0, 'f1': 75.4}","{'acc': 62.5, 'f1': 60.9}","{'acc': 70.5, 'f1': 70.4}","{'acc': 77.9, 'f1': 78.2}"
HateOff,"{'acc': 70.8, 'f1': 58.8}","{'acc': 85.5, 'f1': 66.5}","{'acc': 88.7, 'f1': 68.5}","{'acc': 87.5, 'f1': 68.6}","{'acc': 84.3, 'f1': 66.9}"
HateSpeechX,"{'acc': 54.8, 'f1': 52.0}","{'acc': 50.4, 'f1': 47.7}","{'acc': 65.6, 'f1': 63.9}","{'acc': 56.1, 'f1': 56.6}","{'acc': 49.4, 'f1': 46.1}"
SetFit,"{'acc': 67.7, 'f1': 68.0}","{'acc': 60.5, 'f1': 59.7}","{'acc': 65.3, 'f1': 63.8}","{'acc': 62.6, 'f1': 60.5}","{'acc': 64.3, 'f1': 64.6}"
Mean,"{'acc': 68.0, 'f1': 68.2}","{'acc': 68.7, 'f1': 67.6}","{'acc': 70.1, 'f1': 69.1}","{'acc': 69.5, 'f1': 68.4}","{'acc': 70.1, 'f1': 69.5}"


In [17]:
probs['Base']['probs'][0]

[1.5688783605583012e-11,
 1.5688783605583012e-11,
 1.0,
 5.995204332975845e-15,
 2.3647750424515834e-14,
 4.05634636990726e-10,
 4.274625098332763e-11,
 2.540190280342358e-13,
 1.5688783605583012e-11,
 1.0,
 5.551115123125783e-16,
 1.0,
 1.3154931366443634e-08,
 1.0613732115416497e-13,
 3.979039320256561e-12,
 4.760636329592671e-13,
 7.66053886991358e-15,
 1.2278178473934531e-11,
 3.694822225952521e-13,
 1.0132789611816406e-05,
 1.7139067942650854e-15,
 4.926614671774132e-16,
 5.551115123125783e-16,
 6.110667527536862e-13,
 4.274625098332763e-11,
 1.5788828022778034e-09,
 2.4980018054066022e-15,
 1.5688783605583012e-11,
 1.942890293094024e-15,
 3.979039320256561e-12,
 6.110667527536862e-13,
 3.310560714453459e-10,
 6.110667527536862e-13,
 1.7139067942650854e-15,
 3.441691376337985e-14,
 2.2065682614424986e-15,
 1.5348196029663086e-06,
 7.44648787076585e-12,
 2.2851054382044822e-11,
 5.551115123125783e-16,
 5.115907697472721e-12,
 3.774403012357652e-11,
 6.110667527536862e-13,
 4.418687

In [23]:
for key, value in probs.items():
    if key == 'Llama3.1-70B':
        continue
    for i2, p in enumerate(value['probs']):
        prob_1 = np.array(p[i2], dtype=float)
        prob_1 = np.round(prob_1, 5)
        df_eval_set[key + f"_probs_label_{i2+1}"] = prob_1


In [24]:
df_eval_set.loc[0]

Unnamed: 0                                                             0
text                   Notwithstanding Marriyum Aurangzeb sahiba's po...
ds                                                              berkeley
label_id                                                               3
llama_70B                                                              3
prompt                 <|im_start|>system\nYou are Qwen, created by A...
y_ture                                                                 3
y_pred                                                                 3
Base_probs_label_1                                                   0.0
Base_probs_label_2                                                0.0015
Base_probs_label_3                                                   0.0
Human_probs_label_1                                              0.00433
Human_probs_label_2                                              0.05981
Human_probs_label_3                                